# Supplementary Methods 8-11: statistics

Supplementary Methods 8-11 describe the clinical, regional and cross-omics analyses, the histidine-boundary
analyses, the statistical framework with its testing-family map, and computational provenance. This notebook
recomputes the statistics and counts these sections state, and checks every Benjamini-Hochberg family of the
testing-family map (v28d line 893) by recomputing each adjusted q value within exactly that family.
Run from the repository root with `python paper.py build supplementary_methods_8_11`.

It reads `source_data/supplementary_methods_8_11/` and tables owned by other units (`Fig5`, `Fig6`, `SFig10`,
`SFig11`, `SFig14`, `STab3`, `STab5`), named in each section. There are no figure panels in this unit.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import stats
import pandas as pd
fl.init()
SDR = REPO / "source_data"         # stats functions take the source_data root and read other units' tables by path
pd.set_option("display.width", 200, "display.max_columns", 30)
print(stats.software())

Python 3.12.4; numpy 2.2.6; scipy 1.16.3; pandas 2.3.3; statsmodels 0.14.5; scikit-learn 1.7.2


## CapScan regional human-assigned share

"Two-sided paired sign-flip tests enumerated all 32,768 possible sign patterns across the fifteen participants, with
Benjamini–Hochberg correction across the four regional contrasts" and "95% intervals from 2,000 participant-bootstrap
resamples" (v28d line 843). n = 15 participants, paired (region and stool within participant). Recomputed from the
participant-region origin fractions (`source_data/Fig5/origin_fraction_cells.tsv`) and compared with the producer's
`four_region_tests.tsv`. The bootstrap seed (20261003) is not stated in the text.

In [2]:
rg = stats.stat_regional_human_share(SDR)
print("sign patterns:", rg["patterns"], "| contrasts:", rg["contrasts"], "| max |difference| to producer table:", rg["max_abs_error"])
rg["tests"]

sign patterns: 32768 | contrasts: 4 | max |difference| to producer table: 0.0


,region,n,mean_region,mean_stool,mean_difference,ci_low,ci_high,p,n_positive,statistic_sum,q
0,duodenum,15,24.955218,12.846565,12.108653,6.648172,18.663780,0.001038,12,181.629797,0.002767
1,jejunum,15,23.974396,12.846565,11.127831,5.837149,16.435656,0.002075,12,166.917470,0.002767
2,ileum,15,21.239454,12.846565,8.392889,4.523007,12.528244,0.001404,12,125.893341,0.002767
3,colon,15,17.516291,12.846565,4.669727,0.816404,9.106917,0.037964,9,70.045900,0.037964


## CapScan own-stool permutation test

"Significance was assessed by permuting participant labels of the stool profiles 99,999 times (seed 20260910) ...
two-sided plus-one permutation P value" (v28d line 849); "with every participant closer to their own stool and
P≤1 × 10⁻⁵" (v28d line 847). n = 15 participants, 75 profiles. Recomputed from the 15 x 15 capsule-to-stool distance
matrices (`source_data/supplementary_methods_8_11/own_stool_distance_matrices.tsv`), which reproduce the producer's
own and other distances exactly. No permuted value was as extreme as observed, so P equals its minimum 1/(N+1); the
permutation seed cannot be confirmed from the output. The 95% interval (2,000 participant resamples, seed 20261003)
is not described in the text.

In [3]:
ow = stats.stat_own_stool(SDR, UNIT.name)
print("max |difference| to producer similarity_summary.tsv:", ow["max_abs_error"])
ow["tests"]

max |difference| to producer similarity_summary.tsv: 0.0


,prevalence,participants,own_closer,mean_own,mean_other,mean_advantage,ci_low,ci_high,p,null_exceedances,other_stools
0,0.0,15,15,0.426232,0.486964,0.060732,0.046566,0.075118,0.00001,0,14
1,0.1,15,15,0.425947,0.486643,0.060696,0.046520,0.075108,0.00001,0,14
2,0.2,15,15,0.425514,0.486167,0.060652,0.046494,0.075054,0.00001,0,14


## Cross-omics profile concordance: families

PREDICT: "Benjamini–Hochberg correction was applied across the 28 PREDICT profile-level comparisons" (v28d line 827),
77 patients, 1,999 patient-label permutations. CapScan: "Benjamini–Hochberg correction was applied jointly across all 16
functional-profile comparisons" (v28d line 835), 13 participants, 65 profiles, whole-participant block permutations.
Native profiles: "Benjamini–Hochberg families contained ten or six comparisons" (v28d line 815). Tables:
`source_data/SFig11/functional_correlations_PREDICT.tsv`, `source_data/SFig11/native_correlations_PREDICT.tsv`,
`source_data/Fig5/functional_correlations.tsv`, `source_data/Fig5/native_correlations.tsv` (CapScan on the 347-acquisition
roster). The correlations and permutation P values themselves are recomputed by `figures/Fig5` and `figures/SFig11`.

In [4]:
cr = stats.stat_crossomics(SDR)
print("functional rows:", cr["functional_rows"], "| native rows:", cr["native_rows"], "| max |q error|:", cr["max_abs_q_error"])
print("CapScan mixed sensitivity: non-DNA tests", cr["mixed_nondna_tests"], "on", cr["mixed_nondna_participants"], "participants;",
      "DNA tests", cr["mixed_dna_tests"], "on", cr["mixed_dna_participants"])
print("CapScan functional comparisons changing classification at q<0.05 across filters and populations:", cr["classification_changes"])
display(cr["functional_families"]); display(cr["native_families"])

functional rows: 120 | native rows: 32 | max |q error|: 1.1102230246251565e-16
CapScan mixed sensitivity: non-DNA tests 7 on [15] participants; DNA tests 9 on [13]
CapScan functional comparisons changing classification at q<0.05 across filters and populations: 0


,cohort,scope,prevalence,size,tested,significant,max_abs_q_error
0,CAPSCAN,displayed_mixed12_14,0.0,16,16,16,4.336809e-19
1,CAPSCAN,displayed_mixed12_14,0.1,16,16,16,0.000000e+00
2,CAPSCAN,matched12,0.0,16,16,16,4.336809e-19
3,CAPSCAN,matched12,0.1,16,16,16,0.000000e+00
4,PREDICT,displayed77,0.0,28,28,23,1.110223e-16
5,PREDICT,displayed77,0.1,28,28,23,1.110223e-16


,cohort,analysis,size,tested,significant,max_abs_q_error
0,CAPSCAN,primary_prevalence10,6,6,6,4.336809e-19
1,CAPSCAN,sensitivity_all_features,6,6,6,4.336809e-19
2,PREDICT,primary_prevalence10,10,10,9,1.734723e-18
3,PREDICT,sensitivity_all_features,10,10,9,0.000000e+00


## Controlled distance sensitivity

"Every method/adjustment/input variant used the same design-preserving 1,999 permutations and its own ten- or
six-comparison Benjamini–Hochberg family ... All 176 estimates ... were retained" (v28d line 821). PREDICT 77 patients;
CapScan 13 participants, 65 profiles. Table: `source_data/SFig11/controlled_176_comparisons.tsv`.

In [5]:
co = stats.stat_controlled(SDR)
print("estimates:", co["rows"], "| family sizes:", co["family_sizes"], "| permutations:", co["permutations"], "| max |q error|:", co["max_abs_q_error"])
co["families"]

estimates: 176 | family sizes: [6, 10] | permutations: 1999 | max |q error|: 1.1102230246251565e-16


,cohort,features,distance,adjustment,outer_correlation,size,tested,significant,max_abs_q_error
0,CAPSCAN,all_observed,hellinger,region_adjusted,pearson,6,6,6,0.000000e+00
1,CAPSCAN,all_observed,hellinger,region_adjusted,spearman,6,6,6,0.000000e+00
2,CAPSCAN,all_observed,hellinger,unadjusted,pearson,6,6,6,0.000000e+00
3,CAPSCAN,all_observed,hellinger,unadjusted,spearman,6,6,6,0.000000e+00
4,CAPSCAN,all_observed,rank,region_adjusted,pearson,6,6,4,0.000000e+00
5,CAPSCAN,all_observed,rank,region_adjusted,spearman,6,6,3,9.020562e-17
6,CAPSCAN,all_observed,rank,unadjusted,pearson,6,6,4,1.734723e-18
7,CAPSCAN,all_observed,rank,unadjusted,spearman,6,6,5,0.000000e+00
8,CAPSCAN,prevalence10,hellinger,region_adjusted,pearson,6,6,6,0.000000e+00
9,CAPSCAN,prevalence10,hellinger,region_adjusted,spearman,6,6,6,0.000000e+00


## Individual-function correlations

"Benjamini–Hochberg correction was applied separately within each of the nine representation-by-layer-pair families"
(v28d line 829). Statistical unit patient; at least 35 patients per test. Table:
`source_data/STab5/crossomics_same_function_correlations.tsv`.

In [6]:
ind = stats.stat_individual_functions(SDR)
print("tests:", ind["rows"], "| families:", ind["n_families"], "| minimum patients:", ind["min_patients"], "| max |q error|:", ind["max_abs_q_error"])
ind["families"]

tests: 30581 | families: 9 | minimum patients: 35 | max |q error|: 2.220446049250313e-16


,level,layer_a,layer_b,size,tested,significant,max_abs_q_error
0,KO,metaG,metaT,3995,3995,2816,2.220446e-16
1,KO,metaG,microbial_metaP,4784,4784,163,2.220446e-16
2,KO,metaT,microbial_metaP,3814,3814,101,2.220446e-16
3,OG,metaG,metaT,5796,5796,3954,2.220446e-16
4,OG,metaG,microbial_metaP,6891,6891,138,2.220446e-16
5,OG,metaT,microbial_metaP,4523,4523,79,2.220446e-16
6,Pathway,metaG,metaT,249,249,115,2.220446e-16
7,Pathway,metaG,microbial_metaP,280,280,9,2.220446e-16
8,Pathway,metaT,microbial_metaP,249,249,18,1.110223e-16


## EDS screen

"Benjamini–Hochberg correction was applied separately within each layer, normalization and model to all eligible
features" (v28d line 799); features required positive finite measurements in at least 43 patients. Table:
`source_data/SFig10/all_EDS_associations.tsv`.

In [7]:
sc = stats.stat_eds_screen(SDR)
print("rows:", sc["rows"], "| minimum n among tested features:", sc["min_n_tested"], "| max |q error|:", sc["max_abs_q_error"])
sc["families"]

rows: 49798 | minimum n among tested features: 43 | max |q error|: 2.220446049250313e-16


,layer,normalization,model,size,tested,significant,max_abs_q_error
0,KO,annotated_relative,host_depth_antibiotic_clinical,11878,6965,2,2.220446e-16
1,KO,annotated_relative,unadjusted,11878,6965,324,2.220446e-16
2,KO,native,host_depth_antibiotic_clinical,11878,6965,0,2.220446e-16
3,KO,native,unadjusted,11878,6965,1469,2.220446e-16
4,Metabolite,native,host_depth_antibiotic_clinical,283,283,1,1.110223e-16
5,Metabolite,native,unadjusted,283,283,81,2.220446e-16
6,Pathway,annotated_relative,host_depth_antibiotic_clinical,430,418,9,2.220446e-16
7,Pathway,annotated_relative,unadjusted,430,418,145,2.220446e-16
8,Pathway,native,host_depth_antibiotic_clinical,430,418,0,2.220446e-16
9,Pathway,native,unadjusted,430,418,134,2.220446e-16


## Held-out EDS prediction

"Ridge penalties ... were selected by mean squared error in three inner folds within each of five shuffled
patient-separated outer folds. Three outer split seeds (20260915–20260917) were retained ... 199 label permutations"
(v28d line 803); "Benjamini–Hochberg adjustment across eight grouping-by-model tests ... Two thousand patient
resamples of fixed held-out predictions provided intervals" (v28d line 805). n = 85 patients. The held-out rho,
permutation P, q and both bootstrap intervals are recomputed from the stored predictions and permutation nulls; the
outer folds are regenerated with `KFold(5, shuffle=True, random_state=seed)`. The bootstrap seed (20260918) and the
label-permutation seed (20260915) are not stated in the text.

In [8]:
ed = stats.stat_eds_prediction(SDR, UNIT.name)
print({k: ed[k] for k in ["n", "n_tests", "permutations", "perm_err", "q_err", "rho_err", "ci_err", "inc_err", "fold_ok", "outer_folds", "inner_folds", "seeds", "alphas_selected"]})
ed["tests"][["grouping", "model", "rho", "permutation_p", "p_recomputed", "permutation_q_8", "q_recomputed", "rho_ci_low", "rho_ci_high"]]

{'n': 85, 'n_tests': 8, 'permutations': [199], 'perm_err': 0.0, 'q_err': 3.469446951953614e-18, 'rho_err': 0.0, 'ci_err': 0.0, 'inc_err': 0.0, 'fold_ok': True, 'outer_folds': 5, 'inner_folds': [3], 'seeds': [20260915, 20260916, 20260917], 'alphas_selected': [0.01, 1.0, 10.0, 100.0]}


,grouping,model,rho,permutation_p,p_recomputed,permutation_q_8,q_recomputed,rho_ci_low,rho_ci_high
0,simplified,protein,0.644316,0.005,0.005,0.006667,0.006667,0.487408,0.758877
1,simplified,baseline,0.351385,0.050,0.050,0.050000,0.050000,0.133618,0.537032
2,simplified,combined,0.646446,0.005,0.005,0.006667,0.006667,0.488088,0.756611
3,simplified,distance,0.367351,0.005,0.005,0.006667,0.006667,0.163995,0.553464
12,original,protein,0.644316,0.005,0.005,0.006667,0.006667,0.487408,0.758877
13,original,baseline,0.355332,0.025,0.025,0.028571,0.028571,0.138314,0.545225
14,original,combined,0.649280,0.005,0.005,0.006667,0.006667,0.498127,0.758145
15,original,distance,0.367351,0.005,0.005,0.006667,0.006667,0.163995,0.553464


## PREDICT antibiotic endpoints

"Human-assigned LFQ share and quantified protein-group depth formed one two-test Benjamini–Hochberg family, separately
for the primary definition and each sensitivity definition" (v28d line 795). n = 42 patients (primary). The paired
sign-flip tests and bootstrap intervals need controlled clinical inputs and are recomputed by `tables/STab3`; here the
family structure and the 200,000-draw Monte Carlo grid of the P values are checked
(`source_data/STab3/endpoints_by_definition.tsv`).

In [9]:
ab = stats.stat_antibiotic(SDR)
print("family size:", ab["family_size"], "| P values on the 1/200,001 grid:", ab["on_grid"], "| patients:", ab["n_patients"], "| max |q error|:", ab["max_abs_q_error"])
ab["families"]

family size: 2 | P values on the 1/200,001 grid: True | patients: {'A_current': 42, 'B_HE_as_exposure': 40, 'C_HE_patients_excluded': 35, 'D_HE_off_rows_excluded': 38} | max |q error|: 1.0007186052041206e-16


,scenario,size,tested,significant,max_abs_q_error
0,A_current,2,2,2,9.996344e-17
1,B_HE_as_exposure,2,2,2,1.000177e-16
2,C_HE_patients_excluded,2,2,2,0.000000e+00
3,D_HE_off_rows_excluded,2,2,2,1.000719e-16


## CapScan leading species and Fig. 5e persistence

"Eligibility required a resolved leader in at least twelve of the fifteen participants" (v28d line 785) and "This
30.8% therefore refers only to the 24 displayed functions" (v28d line 787). Tables: `source_data/Fig5/contributor_cells.tsv`,
`source_data/Fig5/mosaic_donor_regions.tsv`.

In [10]:
stats.stat_contributors(SDR)

{'cells': 21168,
 'functions': 1467,
 'participants': 15,
 'min_participants_per_function': 12,
 'mosaic_functions': 24,
 'mosaic_participants': 15,
 'evaluated': 360,
 'persistent': 111,
 'persistence_pct': 30.833333333333332,
 'resolved_rows': 1800}

## DNA runs and native sequence features

"CapScan retained thirteen participants with all five regions, using 245 DNA runs" (v28d line 809); "gene TPM values
from 301 completed metagenome runs" (v28d line 833); "5,391,526 encoded-sequence features had positive RNA abundance,
compared with 14,822,078 for DNA" (v28d line 827). Tables: `capscan_dna_runs.tsv`, `predict_native_feature_summary.tsv`
in this unit's source data.

In [11]:
print(stats.stat_dna_runs(SDR, UNIT.name))
print(stats.stat_native_features(SDR, UNIT.name))

{'functional': 301, 'native': 245, 'native_participants': 13, 'native_complete': True}
{'dna': 14822078, 'rna': 5391526, 'dna_median_retained': 0.2654966471580782, 'rna_median_retained': 0.2099924874638341}


## Supplementary Methods 9: non-specific searches and histidine families

"All 331 CapScan and 306 PREDICT acquisitions were searched" with the standard settings (v28d line 855); the CapScan
count is 347 on the v28f roster. Histidine families (v28d lines 869, 873, 893) are counted and every q value is
recomputed within its family from `source_data/Fig6/`. P values of tests with n ≤ 16 lie on the exact 2^-n grid and
those of PREDICT tests with n = 93 on the 1/200,001 Monte Carlo grid.

In [12]:
display(pd.DataFrame(stats.stat_nonspecific(SDR, UNIT.name)).T)
hi = stats.stat_histidine_families(SDR)
print("exact grid (n<=16):", hi["capscan_exact_grid"], "| Monte Carlo grid (n>16):", hi["predict_mc_grid"], "| PREDICT patients:", hi["predict_patients"], "| max |q error|:", hi["max_abs_q_error"])
hi["families"]

,configured_searches,completed_searches,cleave_at,semi_enzymatic,min_len,max_len,peptide_min_mass,peptide_max_mass,precursor_tol_ppm,fragment_tol_ppm,static_mod_C,variable_mod_M,max_variable_mods,lfq
CAPSCAN,347,347,,False,5,50,500.0,5000.0,20.0,20.0,57.0215,15.9949,2,False
PREDICT,306,306,,False,5,50,500.0,5000.0,20.0,20.0,57.0215,15.9949,2,False


exact grid (n<=16): True | Monte Carlo grid (n>16): True | PREDICT patients: 93 | max |q error|: 1.1102230246251565e-16


,family,size,tested,significant,max_abs_q_error,cohort,table
0,all18_nonKR_composition__length_9,13608,13608,4794,1.110223e-16,CAPSCAN,terminal
1,selection_historical_all_lengths__CAPSCAN,324,324,261,1.110223e-16,CAPSCAN,terminal
2,selection_same_9_50_lengths__CAPSCAN,324,324,282,1.110223e-16,CAPSCAN,terminal
3,selection_same_production_eligibility__CAPSCAN,324,324,279,1.110223e-16,CAPSCAN,terminal
4,terminal_fraction__length_0,756,756,319,1.110223e-16,CAPSCAN,terminal
5,terminal_fraction__length_5,3024,3024,1123,1.110223e-16,CAPSCAN,terminal
6,terminal_fraction__length_9,4536,4536,1695,1.110223e-16,CAPSCAN,terminal
7,H_spatial_all_routes,2520,2520,820,1.110223e-16,CAPSCAN,histidine
8,all_motif_positions_residues,32400,32400,25151,1.110223e-16,CAPSCAN,histidine
9,all_reciprocal_controls,90,90,81,3.469447e-18,CAPSCAN,histidine


## Supplementary Methods 11: healthy-stool grouping counts

"... identical output tables for the 58 acquisitions, including 59,428 quantified protein groups and 1,909,973 quantified
peptide features" (v28d line 909). Values read from the production grouping receipts
(`hstool_grouping_receipt.tsv`); the receipt of the public-module parity check itself was not located (OPEN_ITEMS.md).

In [13]:
stats.stat_hstool_grouping(SDR, UNIT.name)

{'acquisitions': 58,
 'quantified_protein_groups_direct_peptide_assignment': 59428,
 'quantified_protein_groups_original_grouping': 58917,
 'eligible_lfq_features': 1909973,
 'ko_refit_quantified_features': 1909973,
 'ko_refit_positive_acquisitions': 58}

## Checks

There are no quantitative panels in this unit, so there are no pixel checks. Every recomputed number is compared with
the `recomputed_value` column of `NUMBERS.tsv`, and the statistics table is regenerated and compared with `STATS.tsv`.

In [14]:
import csv
print("pixel checks: none (text unit, no panels)")
R, D = stats.collect(SDR, UNIT.name)
print("producer agreement checks:", D["checks"])
want = list(csv.DictReader(open(UNIT / "NUMBERS.tsv", encoding="utf-8"), delimiter="\t"))
got = stats.number_rows(R)
assert len(want) == len(got)
for w, g in zip(want, got):
    assert (w["v28d_line"], w["printed_value"], w["recomputed_value"], w["status"]) == (g["v28d_line"], g["printed_value"], g["recomputed_value"], g["status"]), (w, g)
    print(f'{g["v28d_line"]:>4}  {g["printed_value"]:>26}  {g["recomputed_value"]:>22}  {g["status"]}')
srows = list(csv.DictReader(open(UNIT / "STATS.tsv", encoding="utf-8"), delimiter="\t"))
new = stats.stats_rows(D)
assert len(srows) == len(new) and all({k: a[k] for k in b} == b for a, b in zip(srows, new)), "STATS.tsv differs from the recomputed statistics"
print("STATS.tsv rows reproduced:", len(new))

pixel checks: none (text unit, no panels)


producer agreement checks: {'regional': 0.0, 'own_stool': 0.0, 'crossomics_q': 1.1102230246251565e-16, 'controlled_q': 1.1102230246251565e-16, 'individual_q': 2.220446049250313e-16, 'screen_q': 2.220446049250313e-16, 'antibiotic_q': 1.0007186052041206e-16, 'eds_perm_p': 0.0, 'eds_q': 3.469446951953614e-18, 'eds_rho': 0.0, 'eds_rho_ci': 0.0, 'eds_increment': 0.0, 'histidine_q': 1.1102230246251565e-16}
 783                         80%                          OPEN
 783                         80%                          OPEN
 785                      twelve                  twelve  MATCH
 785                     fifteen                 fifteen  MATCH
 785                         20%                          OPEN
 785                        five                          OPEN
 787                          24                      24  MATCH
 787                     fifteen                 fifteen  MATCH
 787                       30.8%      30.833333333333332  MATCH
 787                    